# Comparação de métodos de detecção de outliers

Este notebook compara **quatro estratégias** para o passo `rm_outliers` do pipeline. Roda os 4 pipelines em paralelo e mostra como cada escolha afeta o clustering, a fronteira de decisão do SVM e a classificação do Sistema Solar.

Um resultado surpreendente: no dataset dos exoplanetas, os métodos ditos "robustos" (MAD) são na verdade os *mais* agressivos em identificar outliers — e isso *piora* o resultado. E o método dito "não robusto" (Z-score) é o único que consegue classificar a Terra como TT — mas por motivo suspeito.

**Requisitos**: rode antes
```bash
python run_pipeline.py --outlier-method iqr    --out outputs        # padrão
python run_pipeline.py --outlier-method zscore --out outputs_zscore
python run_pipeline.py --outlier-method mad    --out outputs_mad
python run_pipeline.py --outlier-method none   --out outputs_none
```

> **Nota sobre execução:** este notebook detecta automaticamente se está rodando no Google Colab ou localmente. No Colab, monta seu Google Drive e assume que a pasta `exoplanetas/` está em `/content/drive/MyDrive/exoplanetas`. Ajuste o caminho na célula abaixo se necessário.

In [ ]:
# ─── Setup: detecta Colab e configura ambiente ────────────────
# Esta célula funciona tanto em Jupyter local quanto no Google Colab.
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # >>> AJUSTE o caminho do projeto no seu Drive se necessário <<<
    PROJECT_ROOT = Path('/content/drive/MyDrive/exoplanetas')
    # instala dependências que o Colab não tem por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'openpyxl'], check=False)
else:
    # Local: assume que este notebook está em <projeto>/notebooks/
    PROJECT_ROOT = Path.cwd().parent

assert PROJECT_ROOT.exists(), f'Projeto não encontrado em: {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Projeto:  {PROJECT_ROOT}')

import warnings; warnings.filterwarnings('ignore')
import io, contextlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from pipeline import preprocessing, diagnostics

DATA = PROJECT_ROOT / "data" / "PSCompData.xlsx"
METODOS = ['iqr', 'zscore', 'mad', 'none']
OUTPUTS = {
    'iqr': PROJECT_ROOT / "outputs",
    'zscore': PROJECT_ROOT / "outputs_zscore",
    'mad': PROJECT_ROOT / "outputs_mad",
    'none': PROJECT_ROOT / "outputs_none",
}

In [ ]:
# Setup específico deste notebook
import io, contextlib, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

METODOS = ['iqr', 'zscore', 'mad', 'none']
OUTPUTS = {
    'iqr':    PROJECT_ROOT / "outputs",
    'zscore': PROJECT_ROOT / "outputs_zscore",
    'mad':    PROJECT_ROOT / "outputs_mad",
    'none':   PROJECT_ROOT / "outputs_none",
}

## Os quatro métodos

| Método | Fórmula | Threshold padrão | Referência da tendência |
|---|---|---|---|
| **IQR** | remove se $x < Q_1 - k \cdot IQR$ ou $x > Q_3 + k \cdot IQR$ | $k=1.5$ | Tukey (1977) |
| **Z-score** | remove se $\left\|\frac{x - \mu}{\sigma}\right\| > k$ | $k=3$ | Regra dos 3σ |
| **MAD** | remove se $\left\|\frac{0.6745 \cdot (x - \tilde{x})}{\text{MAD}}\right\| > k$, onde $\text{MAD} = \text{mediana}(\left\|x - \tilde{x}\right\|)$ | $k=3.5$ | Iglewicz & Hoaglin (1993) |
| **None** | dropna apenas | — | — |

**Intuição comum** (que vamos testar): Z-score é considerado "não robusto" porque $\mu$ e $\sigma$ são puxados pelos próprios outliers. MAD é considerado "robusto" porque usa a mediana. IQR fica no meio.

## 1. Re-executa preprocessing pra ter scaler/PCA em memória

In [ ]:
resultados = {}
for m in METODOS:
    with contextlib.redirect_stdout(io.StringIO()):
        resultados[m] = preprocessing.main(
            caminho_entrada=str(DATA),
            caminho_saida=f"/tmp/_tmp_{m}.xlsx",
            caminho_rgjson=f"/tmp/_tmp_{m}.json",
            outlier_method=m, save_dir=None, show=False,
        )
    print(f"{m:8}: {len(resultados[m]['df_sem_outliers']):,} planetas")

## 2. Panorama numérico

Uma linha por método, com métricas comparáveis.

In [ ]:
# Baseline: quantos planetas passaram até remocao_incertezas
raw = preprocessing.remocao_incertezas(
    preprocessing.def_lines_limits(
        preprocessing.limpeza_dados(
            preprocessing.carregar_arquivo(str(DATA)))))
N_raw = len(raw)

corpos = diagnostics.CORPOS_REFERENCIA
terra = corpos[corpos['nome'] == 'Terra'][diagnostics.COLUNAS_NUMERICAS]

linhas = []
modelos = {}
for m in METODOS:
    df_svm = pd.read_excel(OUTPUTS[m] / "DFsvm.xlsx", index_col=0)
    df_lp = pd.read_excel(OUTPUTS[m] / "DFLabelPropagation.xlsx", index_col=0)
    N = len(df_svm)
    tt = (df_svm['label_lp'] == 1).sum()
    nt = (df_svm['label_lp'] == 0).sum()

    # z-score da Terra com este scaler
    z_terra = resultados[m]['scaler'].transform(terra)[0]

    # menor cluster (proxy de balanceamento)
    menor_cluster = df_lp['cluster_hc'].value_counts().min()

    # SVM treinável?
    if nt >= 2 and tt >= 2:
        feats = [c for c in df_svm.columns if c not in ['pl_name', 'cluster_hc', 'label_lp']]
        M = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=57)
        M.fit(df_svm[feats].values, df_svm['label_lp'].values)
        modelos[m] = M
        pred = 'TT' if M.predict(z_terra.reshape(1, -1))[0] == 1 else 'NT'
    else:
        modelos[m] = None
        pred = '—'

    linhas.append({
        'método': m,
        'N': N,
        '% dataset': f"{100 * N / N_raw:.1f}%",
        'removidos': N_raw - N,
        'TT (LP)': tt,
        'NT (LP)': nt,
        '% TT': f"{100 * tt / N:.1f}%",
        'menor cluster': menor_cluster,
        '|z|_max Terra': f"{abs(z_terra).max():.2f}",
        'Pred. Terra': pred,
    })

resumo = pd.DataFrame(linhas)
print(f"Baseline (após remocao_incertezas): {N_raw:,} planetas\n")
print(resumo.to_string(index=False))

**Leitura:**

- **IQR** remove ~28%, distribui TT/NT razoavelmente (90% TT), mas Terra fica em `|z|=16.7` (fora da distribuição).
- **Z-score** remove só ~5%. Isso é *pouco*: o próprio VHS J1256b infla σ, então o limiar 3σ vira enorme em unidades físicas. Consequência: `|z|` da Terra fica **0.85** (dentro), MAS a distribuição LP colapsa em ~99% TT.
- **MAD** remove ~37% — *mais* que o IQR, porque MAD/mediana são calculados no core central e o limiar 3.5 em unidades físicas fica pequeno. A distribuição LP **inverte** para 19% TT.
- **None** deixa VHS J1256b dominar um cluster inteiro (menor cluster = 1).

## 3. Como cada método classifica o Sistema Solar

In [ ]:
# Predição de cada corpo em cada método
tabela_ss = corpos[['nome', 'tipo_esperado']].copy()

for m in METODOS:
    preds, zmaxes = [], []
    for _, row in corpos.iterrows():
        x_df = pd.DataFrame([row[diagnostics.COLUNAS_NUMERICAS].values],
                            columns=diagnostics.COLUNAS_NUMERICAS)
        z = resultados[m]['scaler'].transform(x_df)[0]
        zmaxes.append(abs(z).max())
        if modelos[m] is not None:
            p = modelos[m].predict(z.reshape(1, -1))[0]
            preds.append('TT' if p == 1 else 'NT')
        else:
            preds.append('—')
    tabela_ss[f'{m}'] = preds
    tabela_ss[f'{m}_|z|'] = [f"{z:.1f}" for z in zmaxes]

print(tabela_ss.to_string(index=False))

**Diagnóstico**: nenhum método está "certo".

- **IQR**: acerta os gigantes (todos NT ✓), mas Vênus/Terra/Marte também saem NT (erro em 3 rochosos).
- **Z-score**: classifica Terra/Vênus/Marte como TT ✓, mas também **Júpiter/Urano/Netuno como TT** ✗ — porque o cluster TT engoliu quase tudo (99% do dataset).
- **MAD**: classifica **tudo como NT** — mesmo a Terra. Cluster NT dominante.
- **None**: SVM inviável.

A aparente "vitória" do Z-score na Terra é ilusória: ele consegue porque tudo é TT. Se o critério fosse "classifica corretamente TT E NT", nenhum método passa.

## 4. Onde a Terra e o Sistema Solar caem em cada espaço PCA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

for ax, m in zip(axes.flat, METODOS):
    df_lp = pd.read_excel(OUTPUTS[m] / "DFLabelPropagation.xlsx", index_col=0)

    for c in sorted(df_lp['cluster_hc'].unique()):
        mask = df_lp['cluster_hc'] == c
        ax.scatter(df_lp.loc[mask, 'PC1'], df_lp.loc[mask, 'PC2'],
                   s=10, alpha=0.4, label=f'Cluster {c} (n={mask.sum()})')

    # Sistema Solar projetado no mesmo espaço PCA
    corpos_df = corpos[diagnostics.COLUNAS_NUMERICAS]
    X_std = resultados[m]['scaler'].transform(corpos_df)
    X_pca = resultados[m]['pca_std'].transform(X_std)

    for i, nome in enumerate(corpos['nome']):
        cor = 'red' if nome == 'Terra' else 'black'
        marker = '*' if nome == 'Terra' else 'x'
        tam = 350 if nome == 'Terra' else 60
        ax.scatter(X_pca[i, 0], X_pca[i, 1], c=cor, marker=marker, s=tam,
                   edgecolors='white', linewidths=1.5, zorder=10)
        ax.annotate(nome, (X_pca[i, 0], X_pca[i, 1]),
                    xytext=(6, 6), textcoords='offset points',
                    fontsize=8, color=cor, fontweight='bold', zorder=11)

    N = len(df_lp)
    ax.set_title(f"{m.upper()} — N={N:,}", fontsize=13, fontweight='bold')
    ax.set_xlabel(f"PC1 ({resultados[m]['pca_std'].explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({resultados[m]['pca_std'].explained_variance_ratio_[1]:.1%})")
    ax.legend(loc='best', fontsize=7)
    ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle("Impacto do método de outlier detection no espaço PCA",
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'figs' / 'comparacao_metodos_outliers.png',
            dpi=120, bbox_inches='tight')
plt.show()

## 5. Distribuição da variável mais problemática (`pl_orbper`)

O período orbital tem a maior amplitude e o pior skew. Vamos ver como cada método corta essa distribuição em escala log.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)

TERRA_ORBPER = 365.25

for ax, m in zip(axes, METODOS):
    df_lp = pd.read_excel(OUTPUTS[m] / "DFLabelPropagation.xlsx", index_col=0)
    # pl_orbper está em z-score no DFLabelPropagation, precisamos desnormalizar
    idx = diagnostics.COLUNAS_NUMERICAS.index('pl_orbper')
    mean = resultados[m]['scaler'].mean_[idx]
    scale = resultados[m]['scaler'].scale_[idx]
    orbper_real = df_lp['pl_orbper'] * scale + mean
    orbper_real = orbper_real[orbper_real > 0]

    ax.hist(np.log10(orbper_real), bins=50, alpha=0.7, color='steelblue',
            edgecolor='white')
    ax.axvline(np.log10(TERRA_ORBPER), color='red', linestyle='--',
               linewidth=2, label=f'Terra ({TERRA_ORBPER:.0f} d)')
    ax.set_title(f"{m.upper()}  (N={len(orbper_real):,})", fontsize=12, fontweight='bold')
    ax.set_xlabel('log₁₀(pl_orbper) [dias]')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim(-1, 7)

axes[0].set_ylabel('Frequência')
plt.suptitle("Distribuição do período orbital após cada filtro (Terra em vermelho)",
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'figs' / 'histogramas_orbper.png',
            dpi=120, bbox_inches='tight')
plt.show()

**Observações**:

- **IQR** e **MAD** cortam a cauda longa (>10³ dias), incluindo a região da Terra (10^2.6 dias). MAD é mais estreito.
- **Z-score** deixa quase tudo passar. A Terra fica dentro da distribuição, mas junto com brown dwarfs de milhões de dias.
- **None** mostra a cauda extrema real.

## 6. Conclusão metodológica

**Contraintuitivo**: no dataset de exoplanetas, a "robustez" do MAD é uma **fraqueza**, não uma força.

A razão é o skew observacional do catálogo. A NASA Exoplanet Archive tem uma cauda longuíssima de períodos por causa de brown dwarfs em órbitas amplas, mas o *core* central é dominado por hot jupiters (períodos < 10 dias). Isso significa que:

- A **mediana** é ~10 dias, o **MAD** é ~5 dias. Threshold 3.5 → planetas com período > ~30 dias são flagged. **Terra (365 dias) é outlier extremo por MAD.**
- A **média** é ~1000 dias (inflada pelos brown dwarfs), o **desvio-padrão** é ~10⁵ dias. Threshold 3 → planetas até ~3×10⁵ dias passam. **Terra é totalmente dentro por Z-score.** Mas o critério é tão frouxo que quase nada é considerado outlier.
- O **IQR** fica no meio porque Q1 e Q3 são menos sensíveis a caudas extremas que média, mas menos sensíveis também que a mediana.

**A lição**: métodos estatísticos agnósticos escolhem o que remover baseado em *estrutura da distribuição*, não em *relevância física*. Num dataset com viés observacional forte, essa diferença é fatal. Nenhum dos quatro dá uma classificação físicamente plausível do Sistema Solar completo.

**Próximos passos possíveis**:

1. **Log-transform antes do outlier filter**: aplicar `log10` em `pl_orbper` e `pl_bmasse` (que têm skew extremo) antes de calcular IQR/MAD. Isso comprime a cauda e deve tornar a Terra menos "outlier" sem colapsar a fronteira.
2. **Filtro físico + outlier estatístico**: primeiro filtra por limites físicos plausíveis (`pl_orbper < 10.000 dias`, `pl_rade < 20 R⊕`), depois aplica MAD sobre esse subset já mais uniforme.
3. **Winsorização**: em vez de remover, comprimir os valores extremos ao 99º percentil. Preserva N mas reduz a influência de VHS J1256b.
4. **Distância Manhattan ou Mahalanobis robusta**: substituir Euclidiana no clustering, o que muda quem é "perto" da Terra sem depender do outlier filter.

Vale destacar no relatório da IC: **a escolha do método de detecção de outliers muda tanto o resultado final quanto os bugs de código encontrados no início**. Isso justifica um capítulo próprio sobre sensibilidade metodológica.